# Unwrapping AI — Zero to Transformer
## Step 07: Cross-Validation — Beginner's Learning Book

A practical, visual, self-contained lesson. You will learn why one train/test split can be misleading, how K-Fold works, how to interpret fold scores, compare models, spot overfitting, and preserve a final test set.

**Dataset:** scikit-learn's built-in Diabetes regression dataset. No separate dataset download is needed.

### Learning outcomes
- Understand folds, K-Fold, validation scores, mean and standard deviation.
- Visualise how rows rotate through training and validation.
- Compare Linear Regression, Ridge and Decision Tree models.
- Interpret fold-score charts and learning curves.
- Avoid data leakage and choose a suitable validation strategy.

## 1. Why cross-validation?

A single train/test split gives one performance estimate. That estimate may be unusually high or low because of which rows happened to land in the test set.

Cross-validation repeats training and validation across different portions of the available data. It provides multiple estimates, helping us understand both typical performance and its variability.

**Vocabulary**
- **Observation/row:** one example in the dataset.
- **Feature (X):** an input variable.
- **Target (y):** the value to predict.
- **Fold:** one partition of the dataset.
- **K-Fold:** divide data into K folds and rotate the validation fold.
- **Validation score:** model performance on the held-out fold.
- **Mean:** average of fold scores.
- **Standard deviation:** how spread out fold scores are.

In ordinary K-Fold, each row is used for validation once and for training in the other K−1 rounds.

## 2. Import libraries

`import` loads a library; `as` gives it a short alias; `from ... import ...` loads selected functions. Parentheses `()` pass arguments, dots `.` access an object's methods, and commas separate arguments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully.")

## 3. Load and inspect data

The Diabetes dataset is included with scikit-learn. It has ten numeric features and a numeric target, so this is a regression task. It is for learning model-validation techniques, not for clinical use.

`X` conventionally represents features; `y` represents the target. `.shape` reports dimensions, `.head()` shows the first five rows, and `.describe()` summarises numeric columns.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.describe())
print("Missing feature values:", X.isna().sum().sum())

## 4. Why one split can be lucky or unlucky

We repeat a random 80/20 split ten times, fit the same Linear Regression model, and record R². R² = 1 is a perfect fit; 0 is equivalent to predicting the target mean; a negative score is worse than that baseline.

`random_state` fixes the random seed so results can be reproduced. This experiment illustrates split sensitivity; it is not our final evaluation protocol.

In [ ]:
single_split_scores = []

for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=seed
    )
    model = Pipeline([
        ("scale", StandardScaler()),
        ("regression", LinearRegression())
    ])
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    single_split_scores.append(r2_score(y_test, predictions))

print("R² scores:", np.round(single_split_scores, 3))
print("Mean:", round(np.mean(single_split_scores), 3))
print("Standard deviation:", round(np.std(single_split_scores), 3))

### Visualisation 1 — variation across random splits

The box plot summarises the spread; the dots show individual split results. A wide spread means the estimate depends noticeably on the particular split.

In [ ]:
plt.figure(figsize=(9, 4))
plt.boxplot(single_split_scores, vert=False)
plt.scatter(single_split_scores, np.ones(len(single_split_scores)), alpha=0.8)
plt.title("R² across 10 different random train/test splits")
plt.xlabel("R² score")
plt.yticks([1], ["Linear Regression"])
plt.grid(axis="x", alpha=0.3)
plt.show()

## 5. How 5-Fold cross-validation works

With five folds, the model is trained and validated five times:
- Round 1: fold 1 validates; folds 2–5 train.
- Round 2: fold 2 validates; folds 1 and 3–5 train.
- Continue until every fold has been the validation fold once.

Each row is validated once. The small example below uses 20 rows so the pattern is easy to see. In the chart, blue means training and orange means validation.

In [ ]:
number_of_rows = 20
demo_splitter = KFold(n_splits=5, shuffle=False)
fold_matrix = np.zeros((5, number_of_rows), dtype=int)

for round_index, (train_idx, valid_idx) in enumerate(demo_splitter.split(np.arange(number_of_rows))):
    fold_matrix[round_index, train_idx] = 1
    fold_matrix[round_index, valid_idx] = 2

from matplotlib.colors import ListedColormap
cmap = ListedColormap(["white", "#4C78A8", "#F58518"])
plt.figure(figsize=(12, 4))
plt.imshow(fold_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=2)
plt.yticks(range(5), [f"Round {i}" for i in range(1, 6)])
plt.xticks(range(20), [f"Row {i}" for i in range(1, 21)])
plt.xlabel("Dataset row")
plt.title("5-Fold rotation: blue = training, orange = validation")
plt.show()

## 6. Create a shuffled K-Fold splitter

`n_splits=5` requests five folds. `shuffle=True` shuffles row order before splitting. `random_state=42` makes the shuffle repeatable.

Shuffling can help when observations are independent and rows have an arbitrary order. Do not blindly use shuffled K-Fold for time-series forecasting: future observations must not be used to predict the past. Use a time-aware splitter such as `TimeSeriesSplit` instead.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_number, (train_idx, valid_idx) in enumerate(kfold.split(X), start=1):
    print(f"Fold {fold_number}: {len(train_idx)} training rows, {len(valid_idx)} validation rows")

`kfold.split(X)` yields pairs of index arrays. `train_idx` identifies training rows; `valid_idx` identifies the held-out validation rows. `enumerate(..., start=1)` numbers the rounds from one. An `f"..."` string inserts values written inside curly braces.

## 7. Run cross-validation

`cross_val_score` automates the repeated fit-and-validate process. We use R² as the metric.

We put `StandardScaler` and Linear Regression inside a `Pipeline`. This is essential: the scaler is fitted only on each fold's training rows. Scaling the entire dataset before cross-validation would leak information from validation rows into training.

In [ ]:
linear_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])

r2_scores = cross_val_score(
    linear_pipeline, X, y, cv=kfold, scoring="r2"
)

print("R² by fold:", np.round(r2_scores, 3))
print("Mean R²:", round(r2_scores.mean(), 3))
print("Standard deviation:", round(r2_scores.std(), 3))

### Interpret the results

A higher mean R² is generally better. Standard deviation describes variation across folds: a smaller value suggests more consistent fold performance. Inspect individual fold scores too; an average can hide one unusually weak fold.

### Visualisation 2 — score in each fold

The dashed line is the mean. A much shorter bar may indicate that one validation fold was harder for the model.

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar([f"Fold {i}" for i in range(1, 6)], r2_scores)
plt.axhline(r2_scores.mean(), linestyle="--", label=f"Mean = {r2_scores.mean():.3f}")
plt.title("Linear Regression: R² across validation folds")
plt.ylabel("R²")
plt.xlabel("Validation fold")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

## 8. Calculate multiple metrics

`cross_validate` can calculate several metrics in one run:
- R²: higher is better.
- MAE: average absolute error; lower is better.
- MSE: average squared error; lower is better.

Scikit-learn represents losses as negative values in its scoring API so that larger scores are always better. We multiply those values by −1 to show familiar positive MAE and MSE.

In [ ]:
multi = cross_validate(
    linear_pipeline, X, y, cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error"
    },
    return_train_score=True
)

summary = pd.DataFrame({
    "Metric": ["R²", "MAE", "MSE"],
    "Mean validation score": [
        multi["test_r2"].mean(),
        -multi["test_mae"].mean(),
        -multi["test_mse"].mean()
    ],
    "Standard deviation": [
        multi["test_r2"].std(),
        multi["test_mae"].std(),
        multi["test_mse"].std()
    ]
})
display(summary.round(3))

## 9. Compare three models on identical folds

We compare Linear Regression, Ridge (linear regression with an L2 penalty), and a Decision Tree (which can learn nonlinear rules). A fair comparison uses the same folds and metric.

Scaling is included for the linear models. A decision tree generally does not require standardisation.

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Decision Tree": DecisionTreeRegressor(max_depth=3, random_state=42)
}

model_scores = {}
rows = []

for name, estimator in models.items():
    scores = cross_val_score(estimator, X, y, cv=kfold, scoring="r2")
    model_scores[name] = scores
    rows.append({"Model": name, "Mean R²": scores.mean(), "Std R²": scores.std()})

comparison = pd.DataFrame(rows).sort_values("Mean R²", ascending=False)
display(comparison.round(3))

### Visualisation 3 — compare model score distributions

Each box represents the five fold scores for one model. Consider both average performance and variability. The highest average score is not automatically the best business choice: interpretability, fairness, stability, operational cost and the problem's error costs matter too.

In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot([model_scores[name] for name in models], tick_labels=list(models.keys()))
plt.title("Cross-validation R² comparison")
plt.ylabel("R² score")
plt.xlabel("Model")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 10. Detect overfitting: training versus validation

`return_train_score=True` returns both training and validation scores for every fold. A high training score paired with a much lower validation score can indicate overfitting. If both scores are low, the model may be underfitting.

A deep decision tree can memorise training observations, so we inspect it here.

In [ ]:
deep_tree = DecisionTreeRegressor(random_state=42)
tree_cv = cross_validate(
    deep_tree, X, y, cv=kfold, scoring="r2", return_train_score=True
)

print("Mean training R²:", round(tree_cv["train_score"].mean(), 3))
print("Mean validation R²:", round(tree_cv["test_score"].mean(), 3))

In [ ]:
rounds = np.arange(1, 6)
plt.figure(figsize=(9, 4))
plt.plot(rounds, tree_cv["train_score"], marker="o", label="Training R²")
plt.plot(rounds, tree_cv["test_score"], marker="o", label="Validation R²")
plt.xticks(rounds)
plt.xlabel("Cross-validation round")
plt.ylabel("R²")
plt.title("Deep Decision Tree: training vs validation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

A large training-validation gap is a warning, not a mathematical proof, of overfitting. Cross-validation helps reveal overfitting; it does not automatically prevent it.

## 11. Learning curves

A learning curve measures performance as the amount of training data increases. It can help diagnose:
- High training score and lower validation score: possible overfitting / high variance.
- Both scores low and close: possible underfitting / high bias.
- Validation score still rising: additional data might help.

This process fits many models and can take longer than a single cross-validation run.

In [ ]:
sizes, train_scores, valid_scores = learning_curve(
    linear_pipeline, X, y, cv=kfold, scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
valid_mean = valid_scores.mean(axis=1)
print("Training sizes:", sizes)
print("Training means:", np.round(train_mean, 3))
print("Validation means:", np.round(valid_mean, 3))

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(sizes, train_mean, marker="o", label="Training score")
plt.plot(sizes, valid_mean, marker="o", label="Cross-validation score")
plt.fill_between(sizes, train_mean-train_scores.std(axis=1), train_mean+train_scores.std(axis=1), alpha=0.15)
plt.fill_between(sizes, valid_mean-valid_scores.std(axis=1), valid_mean+valid_scores.std(axis=1), alpha=0.15)
plt.xlabel("Number of training observations")
plt.ylabel("R²")
plt.title("Learning curve — Linear Regression")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Read the X-axis as the number of training observations and the Y-axis as R². Lines are average scores; shaded areas show roughly one standard deviation. A validation line that is still rising may suggest more data could help. Two low, flat lines may suggest the model or features are too limited.

## 12. Does the number of folds matter?

Five-fold and ten-fold CV are common starting points. Smaller K needs fewer model fits and creates larger validation folds. Larger K requires more computation and creates smaller validation folds. There is no universally best K.

The following cell compares 3, 5 and 10 folds.

In [ ]:
k_rows = []
for k in [3, 5, 10]:
    splitter = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(linear_pipeline, X, y, cv=splitter, scoring="r2")
    k_rows.append({"K folds": k, "Mean R²": scores.mean(), "Std R²": scores.std()})

display(pd.DataFrame(k_rows).round(3))

## 13. Hyperparameter tuning with cross-validation

A hyperparameter is a setting selected before model fitting. Ridge's `alpha` controls the strength of its L2 penalty. Very weak regularisation may not control large coefficients; very strong regularisation may overshrink them and underfit.

We compare several alpha values using the same folds. This is a manual search; `GridSearchCV` and `RandomizedSearchCV` automate similar tasks.

**Important:** Do not report the best tuning score as a final untouched test score. Keep a separate final test set, or use nested cross-validation for a more rigorous estimate.

In [ ]:
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
alpha_rows = []

for alpha in alpha_values:
    ridge = Pipeline([
        ("scale", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])
    scores = cross_val_score(ridge, X, y, cv=kfold, scoring="r2")
    alpha_rows.append({"Alpha": alpha, "Mean R²": scores.mean(), "Std R²": scores.std()})

alpha_table = pd.DataFrame(alpha_rows)
display(alpha_table.round(3))

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(alpha_table["Alpha"], alpha_table["Mean R²"], marker="o")
plt.xscale("log")
plt.xlabel("Ridge alpha (log scale)")
plt.ylabel("Mean cross-validation R²")
plt.title("Ridge alpha tuning with cross-validation")
plt.grid(alpha=0.3)
plt.show()

## 14. A safe final evaluation workflow

1. Set aside a final test set once.
2. Use only the remaining training data for cross-validation and model selection.
3. Choose the model and hyperparameters.
4. Refit the selected pipeline on all training data.
5. Evaluate once on the untouched final test set.

Repeatedly checking the final test score while tuning makes it part of the model-selection process and weakens its independence.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

final_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

training_cv = cross_val_score(
    final_model, X_train, y_train,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="r2"
)

final_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

print("Training-only CV mean R²:", round(training_cv.mean(), 3))
print("Untouched test R²:", round(r2_score(y_test, test_predictions), 3))
print("Test MAE:", round(mean_absolute_error(y_test, test_predictions), 3))
print("Test MSE:", round(mean_squared_error(y_test, test_predictions), 3))

## 15. Common mistakes

1. **Scaling before cross-validation:** fit preprocessing inside a Pipeline to avoid leakage.
2. **Repeatedly using the final test set:** it becomes part of tuning and is no longer an independent check.
3. **Shuffled K-Fold for time series:** use a time-aware split.
4. **Ignoring groups:** records from one customer, patient or device may leak across folds. Use group-aware splitting when needed.
5. **Reporting only the mean:** show variability or individual fold scores too.
6. **Comparing models on different folds:** use the same split strategy for fair comparison.
7. **Assuming cross-validation guarantees future performance:** it estimates performance under assumptions; it is not a guarantee.

## 16. Industry example: credit-risk modelling

Suppose we estimate the probability that a loan applicant will default. Cross-validation can compare candidate models and reveal variation across validation samples.

But split design must match deployment:
- Predicting future applicants? Consider time-based validation.
- Multiple records per customer? Keep customer groups separated.
- Deploying across branches or countries? Test whether validation folds represent those settings.
- Evaluate more than one metric: discrimination, calibration, error costs, fairness and stability can all matter.
- Exclude post-outcome and future information from features.

Cross-validation is one component of model validation and governance, not a substitute for them.

## 17. Practice exercises

1. Change `n_splits=5` to 10. How many rounds run?
2. Set `shuffle=False`. What changes, and why could row ordering matter?
3. Compare Ridge alpha values 0.01, 1 and 100. Does stronger regularisation always improve the score?
4. Try Decision Tree `max_depth` values 2, 4, 8 and `None`. Compare training and validation scores.
5. Explain why the final test set should not be used to tune hyperparameters.

## 18. Suggested answers

1. Ten folds create ten rounds. Each row is validated once.
2. Consecutive rows remain together. If rows are ordered by date, target or another pattern, folds may not represent the same distribution.
3. No. Very weak regularisation may not help; very strong regularisation may underfit. The useful value depends on the data.
4. Increasing depth lets a tree fit training data more closely; validation performance may eventually deteriorate from overfitting. Run the experiment and compare.
5. Repeated decisions based on the final test score indirectly tune to that set, making the final estimate optimistic.

## 19. Knowledge check

1. In 5-Fold CV, how many times is each row used for validation?
2. What does the standard deviation of fold scores tell you?
3. Why should a scaler be inside a pipeline?
4. Is shuffled K-Fold appropriate for forecasting future time-series values?
5. Should the final test set be used to tune Ridge alpha?

**Answers:** 1) Once. 2) Variation across folds. 3) To fit scaling only on each training fold and avoid leakage. 4) Usually no; use time-aware validation. 5) No; tune using training data and CV.

## 20. Glossary

- **Cross-validation:** repeated train/validation evaluation on different data partitions.
- **Fold:** one partition of the dataset.
- **K-Fold:** CV with K partitions.
- **Data leakage:** improper use of information from outside the training portion.
- **Pipeline:** preprocessing and model steps fitted together.
- **R²:** coefficient of determination.
- **MAE:** mean absolute error.
- **MSE:** mean squared error.
- **Hyperparameter:** a setting selected outside the model's parameter-fitting process.
- **Nested CV:** outer evaluation loop around an inner model-selection loop.
- **TimeSeriesSplit:** splitter respecting temporal order.
- **GroupKFold:** splitter keeping groups separated across training and validation.

## Completion checklist
- [ ] I can explain why one split may be unreliable.
- [ ] I can explain folds and K-Fold.
- [ ] I understand `n_splits`, `shuffle` and `random_state`.
- [ ] I can interpret fold scores, mean and standard deviation.
- [ ] I can compare models on identical folds.
- [ ] I understand preprocessing leakage and Pipelines.
- [ ] I know when ordinary K-Fold is unsuitable.
- [ ] I know why the final test set must remain untouched during tuning.

**Suggested GitHub filename:** `Step_07_Cross_Validation_Complete_Guide.ipynb`  
**Suggested commit:** `Add Step 07 - Cross-Validation complete beginner guide`

**Next:** Step 08 — Decision Trees, with tree diagrams and practical visualisations.